# SB System - Windows MT5 Candle Export

Use this notebook on the Windows laptop/VPS where MetaTrader 5 is installed and logged in.

This notebook does **not** require Docker or PostgreSQL. It exports MT5 candles from `SB_IMPORT_START` onward into compressed CSV files under `data/raw/mt5_export`, then you can copy that folder to the MacBook.

## 1. Load Project Configuration

Before running this notebook, update `.env` with the broker symbol names you want to export. Broker names may include suffixes such as `EURUSDm`, `XAUUSD.a`, or `US30.cash`.

In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sb_system.market_data import load_config

config = load_config(PROJECT_ROOT / ".env", require_database_url=False)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Symbols: {config.symbols}")
print(f"Timeframes: {config.timeframes}")
print(f"Import start: {config.import_start}")

## 2. Verify MetaTrader 5 Connection

MetaTrader 5 must be open and logged in before running this cell.

In [ ]:
try:
    import MetaTrader5 as mt5
except ImportError as exc:
    raise ImportError(
        "MetaTrader5 package is not installed in this notebook kernel. "
        f"Run: {sys.executable} -m pip install -r requirements-win-mt5.txt"
    ) from exc

if not mt5.initialize():
    raise RuntimeError(f"MT5 initialize failed: {mt5.last_error()}")

account_info = mt5.account_info()
terminal_info = mt5.terminal_info()

print("MT5 initialized")
print(f"Account: {account_info.login if account_info else 'unknown'}")
print(f"Terminal path: {terminal_info.path if terminal_info else 'unknown'}")

mt5.shutdown()

## 3. Export Candles to CSV Files

This calls `scripts/export_mt5_candles.py`, which pulls each configured symbol/timeframe and writes `.csv.gz` files.

In [ ]:
output_dir = PROJECT_ROOT / "data" / "raw" / "mt5_export"
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "export_mt5_candles.py"),
    "--output-dir",
    str(output_dir),
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_ROOT)
if result.returncode != 0:
    raise RuntimeError(f"MT5 export failed with exit code {result.returncode}")

print(f"Export folder: {output_dir}")

## 4. Inspect Export Manifest

In [ ]:
manifest_path = output_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print(f"Created at: {manifest['created_at']}")
print(f"Date range: {manifest['date_from']} -> {manifest['date_to']}")
print(f"Total rows: {manifest['total_rows']}")

pd.DataFrame(manifest["files"])

## 5. Preview One Exported File

In [ ]:
csv_files = sorted(output_dir.glob("*.csv.gz"))
if not csv_files:
    raise FileNotFoundError(f"No exported CSV files found in {output_dir}")

preview_file = csv_files[0]
print(preview_file)
pd.read_csv(preview_file).head(20)

## 6. Move Files to MacBook

Copy the whole folder below to the same path in the MacBook project:

```text
data/raw/mt5_export
```

Then run `notebooks/02_mac_import_csv_to_postgres.ipynb` on the MacBook.